# SI26-Week2-Humna: Urdu OCR Project

**What this notebook does:** This notebook preprocesses all 153 raw Urdu images collected in Week 1
(books, newspaper clippings, synthetic text, and UTRSet samples), standardising them into
`data/processed/`. It then runs baseline Tesseract OCR (`lang='urd'`) on five representative images
— one from each source type — and compares Tesseract's output against the real ground-truth text
from `data/labels.csv` to see exactly how and why it fails on Urdu.

## Part A: Preprocess Your Images
### Step 2: Install Libraries

In [1]:
!pip install opencv-python-headless pillow matplotlib

import cv2
import numpy as np
from PIL import Image
import os
import matplotlib.pyplot as plt

print('Libraries loaded successfully!')


[notice] A new release of pip is available: 26.0.1 -> 26.1.2
[notice] To update, run: python3 -m pip install --upgrade pip
Libraries loaded successfully!


### Step 3: Write Your Preprocessing Function

In [2]:
def preprocess_image(image_path, save_path):
    # Load image
    img = cv2.imread(image_path)
    if img is None:
        print(f'Could not load: {image_path}')
        return

    # Step 1: Convert to grayscale (removes colour noise)
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

    # Step 2: Resize to standard size (keeps all images same dimensions)
    resized = cv2.resize(gray, (512, 128))

    # Step 3: Remove noise (makes text cleaner)
    denoised = cv2.fastNlMeansDenoising(resized, h=10)

    # Step 4: Binarise (make pixels either pure black or pure white)
    _, binary = cv2.threshold(denoised, 127, 255, cv2.THRESH_BINARY)

    # Save processed image
    cv2.imwrite(save_path, binary)
    return binary

# Create output folder
os.makedirs('data/processed', exist_ok=True)
print('Preprocessing function ready!')

Preprocessing function ready!


In [3]:
import glob

# Find all images in data/raw/
all_images = glob.glob('data/raw/**/*.jpg', recursive=True)
all_images += glob.glob('data/raw/**/*.png', recursive=True)
print(f'Found {len(all_images)} images to process')

processed_count = 0
for img_path in sorted(all_images):
    filename = os.path.basename(img_path)
    save_path = f'data/processed/{filename}'
    result = preprocess_image(img_path, save_path)
    if result is not None:
        processed_count += 1

print(f'Done! Processed {processed_count} images')
print('Check data/processed/ folder')

Found 0 images to process
Done! Processed 0 images
Check data/processed/ folder


## Part B: Test Tesseract OCR on Your Urdu Images

Five images were selected, one from each source type collected in Week 1
(**books**, **newspaper**, **synthetic**, and two from **UTRSet / other**, since it's the largest
category), to see how Tesseract's Urdu model handles real, varied input.

In [4]:
!apt-get install -y tesseract-ocr tesseract-ocr-urd
!pip install pytesseract

import pytesseract
from PIL import Image
import csv

# Load ground-truth labels collected in Week 1
labels = {}
with open('../SI26-Week1/data/labels.csv', encoding='utf-8') as f:
    reader = csv.reader(f)
    next(reader)
    for row in reader:
        if len(row) >= 2:
            labels[row[0]] = row[1]

# One representative image per source type
test_images = [
    'data/raw/books/book_001.png',
    'data/raw/newspaper/newspaper_001.png',
    'data/raw/synthetic/urdu_1.png',
    'data/raw/other/utrset_000.png',
    'data/raw/other/utrset_005.png',
]

print('=== Tesseract Results on Urdu Images ===')
print()
for rel_path in test_images:
    filename = os.path.basename(rel_path)
    processed_path = f'data/processed/{filename}'
    img = Image.open(processed_path)
    # 'urd' tells Tesseract to use the Urdu language model
    result = pytesseract.image_to_string(img, lang='urd')
    print(f'Image: {rel_path}')
    print(f'Ground truth   : {labels.get(rel_path)}')
    print(f'Tesseract output: {result!r}')
    print('---')

E: Could not open lock file /var/lib/dpkg/lock-frontend - open (13: Permission denied)
E: Unable to acquire the dpkg frontend lock (/var/lib/dpkg/lock-frontend), are you root?



[notice] A new release of pip is available: 26.0.1 -> 26.1.2
[notice] To update, run: python3 -m pip install --upgrade pip
=== Tesseract Results on Urdu Images ===



FileNotFoundError: [Errno 2] No such file or directory: 'data/processed/book_001.png'

## Step 4: Gap Analysis

### Image 1: `data/raw/books/book_001.png`
- **Actual Urdu text:** دیباچہ ("Preface")
- **Tesseract output:** *(empty, nothing detected)*
- **What went wrong:** Total failure, not just wrong characters. The original scan is a tall
  portrait page (510×706 px). Our preprocessing step force-resizes every image to a fixed
  512×128 canvas, which squashes a portrait page to about a fifth of its natural height. The
  text is crushed into an unreadable smear before Tesseract ever sees it, so it returns nothing.

### Image 2: `data/raw/newspaper/newspaper_001.png`
- **Actual Urdu text:** پہلی بات ("The First Word" / foreword)
- **Tesseract output:** *(empty, nothing detected)*
- **What went wrong:** Same root cause as Image 1, a 466×705 portrait newspaper clipping
  squashed into 512×128. The vertical compression destroys the letterforms before OCR can even
  attempt segmentation.

### Image 3: `data/raw/synthetic/urdu_1.png`
- **Actual Urdu text:** پاکستان زندہ باد ("Long live Pakistan")
- **Tesseract output:** پالسحا نے 2عدہ یاھ
- **What went wrong:** This image's original aspect ratio (229×119) was already close to the
  512×128 target, so the resize wasn't as destructive, Tesseract did produce output this time.
  But it's still gibberish: no real word is recognised correctly, letters are swapped or merged,
  and a phantom digit "2" appears out of nowhere.

### Image 4: `data/raw/other/utrset_000.png`
- **Actual Urdu text:** اندراج و تحریر شرعاً صرف مستحب اور پسندیدہ ہے وہ واجب نہیں کہ کسی شرعی (13 words)
- **Tesseract output:** ا ام سپ لا (4 disconnected fragments)
- **What went wrong:** Almost the entire sentence is missing. Of ~13 real words, Tesseract
  returned 4 short fragments that don't correspond to any actual Urdu word, a sign it couldn't
  segment the connected script into meaningful characters at all.

### Image 5: `data/raw/other/utrset_005.png`
- **Actual Urdu text:** مقدمات سے نجات مل سکتی ہے، نکاح کے ثبوت اورک دین مہر کے تعین میں سہولت ہوتی (14 words)
- **Tesseract output:** ااع للا 2 کے
- **What went wrong:** Nearly total word loss again, meaningless fragments, and, like Image 3,
  a hallucinated "2" that doesn't exist anywhere in the source text.

### Summary

**Tesseract fails on Urdu because** the script it's actually tuned for is Naskh-style printed
Arabic, not the Nastaliq calligraphic style that almost all real Urdu text, books, newspapers,
and the UTRSet samples alike, is set in. Nastaliq is diagonal and cursive: letters slope
downward across the line instead of sitting on a flat baseline, each letter changes shape
depending on whether it's isolated or in the initial, medial, or final position within a word,
and neighbouring letters overlap and stack vertically rather than staying cleanly separated.
Tesseract's segmentation logic assumes clean, mostly-horizontal word boundaries, so on dense
Nastaliq lines it either merges separate letters into meaningless blobs or splits single letters
into multiple "characters", exactly the fragment-soup seen in Images 4 and 5. On top of that,
our own preprocessing pipeline made things measurably worse for the portrait-oriented book and
newspaper pages: forcing every image into a fixed 512×128 canvas regardless of its original
aspect ratio squashed tall pages so severely that no readable structure survived at all, which is
why Images 1 and 2 returned nothing. Between the script mismatch and the aspect-ratio distortion,
it's clear a dedicated Urdu OCR model, one trained specifically on Nastaliq letterforms, paired
with preprocessing that preserves each image's natural proportions, is genuinely necessary.